In [8]:
%load_ext autoreload
%autoreload 2

from datetime import datetime, timedelta, date
from functools import partial
import numpy as np
import polars as pl

from okx.store import OrderbookStore
from okx.recipes.options import prepare_options, build_forwards_options_comparison
from okx.recipes.forwards import build_forwards_pchip, build_forwards_kalman, prepare_pillars

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
store = OrderbookStore(
    data_root="data/okx",
    manifest_path="data/okx/manifest.sqlite",
    batch_days=5
)

In [10]:
store.clear_cache()

Cleared all caches


In [11]:
start_date = date(2025, 9, 1)
end_date = date(2025, 9, 2)
dates = [start_date + timedelta(days=i) for i in range((end_date - start_date).days + 1)]

print(f"Testing with dates: {dates[0]} to {dates[-1]}")

# just one date for now
dates = [date(2025, 9, 2)]


Testing with dates: 2025-09-01 to 2025-09-02


In [12]:
options_df = prepare_options(
    store,
    inst_family='BTC-USD',
    dates=dates,
    forwards_recipe=build_forwards_kalman,
    binning=None
)

Time taken to fetch options: 0:00:10.551377
Options dataset stats -> rows: 29,216,457, unique timeMs: 6,312,288, unique expiries: 12
Passing 6,312,288 unique option timestamps to forwards recipe


<sys>:0: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided


Time taken to prepare pillars: 0:00:08.893272
Time taken to build snapshots: 0:00:45.502213 (6,305,843 usable timestamps)
Time taken to run Kalman filter: 0:04:57.778788
Time taken to convert states: 0:03:22.574083
Time taken to fetch forwards: 0:10:28.664813
Forward dataset stats -> rows: 6,305,843, unique curve timestamps: 6,305,843
Built forward lookup for 6,305,843 timestamps in 0:00:22.625426


Matching forwards to options: 100%|██████████| 6312288/6312288 [01:48<00:00, 58255.72it/s]


Skipped 6,445 timestamp groups with no forward curve
Time taken to match forwards to options: core 0:01:55.305929, incl. setup 0:02:18.995973 (matched 29,187,548 / 29,216,457 options)


In [13]:
print(options_df.shape)
print(options_df.head())
options_df_filtered = options_df.filter((pl.col('bid_1_px').is_not_null()) & (pl.col('ask_1_px').is_not_null()))
print(options_df_filtered.shape)
print(options_df_filtered.head())
duplicate_counts = (
    options_df_filtered
    .group_by([col for col in options_df_filtered.columns])
    .count()
    .filter(pl.col("count") > 1)
    .sort("count", descending=True)
)

print("Duplicate rows and their counts:")
print(duplicate_counts)


(29216457, 11)
shape: (5, 11)
┌────────────┬────────────┬──────────┬──────────┬───┬──────────┬───────────┬───────────┬───────────┐
│ timeMs     ┆ symbol     ┆ bid_1_px ┆ ask_1_px ┆ … ┆ opt_type ┆ F_bid     ┆ F_ask     ┆ moneyness │
│ ---        ┆ ---        ┆ ---      ┆ ---      ┆   ┆ ---      ┆ ---       ┆ ---       ┆ ---       │
│ i64        ┆ str        ┆ f64      ┆ f64      ┆   ┆ str      ┆ f64       ┆ f64       ┆ f64       │
╞════════════╪════════════╪══════════╪══════════╪═══╪══════════╪═══════════╪═══════════╪═══════════╡
│ 1756771507 ┆ BTC-USD-25 ┆ 0.005    ┆ null     ┆ … ┆ C        ┆ 109120.72 ┆ 109120.82 ┆ 0.916416  │
│ 999        ┆ 0903-10000 ┆          ┆          ┆   ┆          ┆ 9946      ┆ 9971      ┆           │
│            ┆ 0-C.OK     ┆          ┆          ┆   ┆          ┆           ┆           ┆           │
│ 1756771519 ┆ BTC-USD-25 ┆ 0.005    ┆ null     ┆ … ┆ C        ┆ 109120.72 ┆ 109120.82 ┆ 0.916416  │
│ 307        ┆ 0903-10000 ┆          ┆          ┆   ┆        

/var/folders/fl/fdwpgpsx7fx15p__07t93bhr0000gn/T/ipykernel_28342/3014190339.py:9: DeprecationWarning: `GroupBy.count` was renamed; use `GroupBy.len` instead
  .count()


Duplicate rows and their counts:
shape: (0, 12)
┌────────┬────────┬──────────┬──────────┬───┬───────┬───────┬───────────┬───────┐
│ timeMs ┆ symbol ┆ bid_1_px ┆ ask_1_px ┆ … ┆ F_bid ┆ F_ask ┆ moneyness ┆ count │
│ ---    ┆ ---    ┆ ---      ┆ ---      ┆   ┆ ---   ┆ ---   ┆ ---       ┆ ---   │
│ i64    ┆ str    ┆ f64      ┆ f64      ┆   ┆ f64   ┆ f64   ┆ f64       ┆ u32   │
╞════════╪════════╪══════════╪══════════╪═══╪═══════╪═══════╪═══════════╪═══════╡
└────────┴────────┴──────────┴──────────┴───┴───────┴───────┴───────────┴───────┘


In [14]:
def show_df_time_range(name, df):
    if df.is_empty():
        print(f"{name}: DataFrame is empty")
        return
    timeMs = df['timeMs'].to_numpy()
    earliest = timeMs.min()
    latest = timeMs.max()
    if hasattr(earliest, 'item'):
        earliest = earliest.item()
    if hasattr(latest, 'item'):
        latest = latest.item()
    earliest_dt = datetime.utcfromtimestamp(earliest / 1000)
    latest_dt = datetime.utcfromtimestamp(latest / 1000)
    print(f"{name}:")
    print(f"  Earliest timeMs: {earliest} ({earliest_dt})")
    print(f"  Latest   timeMs: {latest} ({latest_dt})\n")

show_df_time_range("options_df", options_df)

options_df:
  Earliest timeMs: 1756771200017 (2025-09-02 00:00:00.017000)
  Latest   timeMs: 1756857599990 (2025-09-02 23:59:59.990000)



/var/folders/fl/fdwpgpsx7fx15p__07t93bhr0000gn/T/ipykernel_28342/3855728042.py:12: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  earliest_dt = datetime.utcfromtimestamp(earliest / 1000)
/var/folders/fl/fdwpgpsx7fx15p__07t93bhr0000gn/T/ipykernel_28342/3855728042.py:13: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  latest_dt = datetime.utcfromtimestamp(latest / 1000)


In [15]:
options_df.head()

timeMs,symbol,bid_1_px,ask_1_px,expiry,T,strike,opt_type,F_bid,F_ask,moneyness
i64,str,f64,f64,i64,f64,i64,str,f64,f64,f64
1756771507999,"""BTC-USD-250903-100000-C.OK""",0.005,null,1756886400000,0.003643,100000,"""C""",109120.729946,109120.829971,0.916416
1756771519307,"""BTC-USD-250903-100000-C.OK""",0.005,null,1756886400000,0.003643,100000,"""C""",109120.721674,109120.821699,0.916416
1756772419535,"""BTC-USD-250903-100000-C.OK""",0.005,null,1756886400000,0.003614,100000,"""C""",109189.242346,109189.342371,0.915841
1756772441613,"""BTC-USD-250903-100000-C.OK""",0.005,null,1756886400000,0.003614,100000,"""C""",109189.237566,109189.337591,0.915841
1756773153239,"""BTC-USD-250903-100000-C.OK""",0.005,null,1756886400000,0.003591,100000,"""C""",109019.040071,109019.140096,0.917271
